# SILVER LAYER

# Various Imports and Python Environment Configuration. Create the spark session.

In [1]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_DIR = PROJECT_ROOT / "delta" / "bronze"
SILVER_DIR = PROJECT_ROOT / "delta" / "silver"

SILVER_DIR.mkdir(parents=True, exist_ok=True)

builder = (
    SparkSession.builder
    .appName("silver-transformation")
    .master("local[*]")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension",
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.sql.shuffle.partitions", "8")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Python      :", sys.executable)
print("Bronze      :", BRONZE_DIR)
print("Silver      :", SILVER_DIR)

Python      : C:\lufthansa-de-exercise\.venv\Scripts\python.exe
Bronze      : C:\lufthansa-de-exercise\delta\bronze
Silver      : C:\lufthansa-de-exercise\delta\silver


# Simple checking if every table is present for the silver layer.

In [2]:
all_bronze_tables = [
    "customers",
    "products",
    "sellers",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
]

missing_tables = []

for name in all_bronze_tables:
    if not (BRONZE_DIR / name / "_delta_log").exists():
        missing_tables.append(name)

if missing_tables:
    raise FileNotFoundError(
        "Missing Bronze Delta tables: " + ", ".join(missing_tables)
    )

print("All Bronze Delta tables required are available.")

All Bronze Delta tables required are available.


# Create functions to read the bronze table and write in the silver tables.

In [3]:
PARTITION_COLS = ["year", "month", "day"]
write_results = {}

def read_bronze(table_name):
    path = BRONZE_DIR / table_name
    return spark.read.format("delta").load(str(path))

def write_silver(df, table_name, partitioned=True):
    output_path = SILVER_DIR / table_name
    # check if the table is partitioned and also added an addition if there is any partitioned column missiong
    if partitioned:
        missing = [col for col in PARTITION_COLS if col not in df.columns]
        if missing:
            raise ValueError(
                f"{table_name} cannot be partitioned; missing columns: {missing}"
            )

    row_count = df.count()
    
    # write the data into delta lakes while maintaing the same format of partitioning by year, month, day
    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )

    if partitioned:
        writer = writer.partitionBy(*PARTITION_COLS)

    # save the results in the silver path and contain the path, count and partition in a dictionary
    writer.save(str(output_path))

    write_results[table_name] = {
        "output_path": str(output_path),
        "row_count": row_count,
        "partitioned": partitioned,
    }

    print(
        f"{table_name:<22} rows={row_count:>8,} "
        f"partitioned={partitioned}"
    )

    return df

# Started using dropna handling null values in orders table.

In [4]:
orders_bronze = read_bronze("orders")

# decided to approach the process in 2 parts:
# doing a complete table null handling and doing a custom null handling in the main columns
count_orders = orders_bronze.count()
count_dropna_orders = orders_bronze.dropna().count()
count_custom_dropna_orders = orders_bronze.dropna(
    subset=["order_id", "customer_id", "order_purchase_timestamp"]
).count()

print(f"Total orders                 : {count_orders:,}")
print(f"After dropna process         : {count_dropna_orders:,}")
print(f"After custom dropna process  : {count_custom_dropna_orders:,}")
print(f"Difference of rows           : {count_orders - count_dropna_orders:,}")

Total orders                 : 99,441
After dropna process         : 96,461
After custom dropna process  : 99,441
Difference of rows           : 2,980


# Check 3 main reference tables regarding dropna and dropDuplicates and write them in silver layer.

In [5]:
"""
There is quite a difference in using dropna for a full table, as we can lose valuable data. My approach regarding this topic goes like this:
These tables customers, sellers, order_items, order_payments dont have any null values. 
The others orders have order_approved, order_delivered_carrier_date and order_delivered_customer_date, 
but these are meaningful data even though empty, so we cant remove them. Same thing with order_reviews. 
The only one needed attention is in products table named product_category_name which contains lots of null values but still valuable information. 
There i decided to use coalesce in order for these rows to have a better use in gold layer.
"""

columns = {
    "customers": "customer_id",
    "products": "product_id",
    "sellers": "seller_id",
}

silver_wr = {}

for table_name, key_column in columns.items():
    df = (
        read_bronze(table_name)
        .dropna(subset=[key_column])
        .dropDuplicates([key_column])
    )

    if table_name == "products":
        df = df.withColumn(
            "product_category_name",
            F.coalesce(F.col("product_category_name"), F.lit("unknown")),
        )

    silver_wr[table_name] = write_silver(df, table_name, partitioned=False)

customers              rows=  99,441 partitioned=False
products               rows=  32,951 partitioned=False
sellers                rows=   3,095 partitioned=False


# Check orders table regarding dropna and dropDuplicates and write it in silver layer.

In [6]:
silver_orders = (
    read_bronze("orders")
    .withColumn("order_purchase_timestamp", F.col("order_purchase_timestamp").cast(T.TimestampType()))
    .withColumn("order_delivered_customer_date", F.col("order_delivered_customer_date").cast(T.TimestampType()))
    .withColumn("delivery_time_days", F.datediff(F.to_date("order_delivered_customer_date"), F.to_date("order_purchase_timestamp"))) # requested calculated column
    .dropna(subset=["order_id", "customer_id", "order_purchase_timestamp"])
    .dropDuplicates(["order_id"])
    .withColumn("year", F.year("order_purchase_timestamp"))
    .withColumn("month", F.month("order_purchase_timestamp"))
    .withColumn("day", F.dayofmonth("order_purchase_timestamp"))
)

silver_orders = write_silver(silver_orders, "orders")

orders                 rows=  99,441 partitioned=True


# Check order items table regarding dropna and dropDuplicates and write it in silver layer.

In [7]:
silver_order_items = (
    read_bronze("order_items")
    .withColumn("price", F.col("price").cast(T.DoubleType()))
    # Ccasting columns to appopriate types for later calculation
    .withColumn("freight_value", F.col("freight_value").cast(T.DoubleType()))
    .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast(T.TimestampType()))
    .withColumn("total_price", F.round(F.col("price") + F.col("freight_value"), 2)) # requested calculated column
    .withColumn("profit_margin", F.round(F.expr("price - freight_value"), 2)) # requested calculated column
    .dropna(
        subset=["order_id", "order_item_id", "price", "freight_value"])
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("year", F.year("shipping_limit_date"))
    .withColumn("month", F.month("shipping_limit_date"))
    .withColumn("day", F.dayofmonth("shipping_limit_date"))
)

silver_order_items = write_silver(silver_order_items, "order_items")

order_items            rows= 112,650 partitioned=True


In [8]:
# example from the orders item table
silver_order_items.select(
    "order_id",
    "order_item_id",
    "price",
    "freight_value",
    "total_price",
    "profit_margin",
).show(5, truncate=False)

+--------------------------------+-------------+-----+-------------+-----------+-------------+
|order_id                        |order_item_id|price|freight_value|total_price|profit_margin|
+--------------------------------+-------------+-----+-------------+-----------+-------------+
|02dcfbe3274b16820a9fca74c156dc9a|1            |454.0|15.52        |469.52     |438.48       |
|04f89a77d1fcc3178c1a374b5cc61a52|1            |49.9 |9.34         |59.24      |40.56        |
|075434afaadfb0809ecf3f095a372340|1            |42.99|9.34         |52.33      |33.65        |
|0869a3c8a1dd07c13e4847627fcc704f|1            |16.9 |15.1         |32.0       |1.8          |
|090c0cc3d47ff02fedac32aa0a7cab97|1            |59.9 |19.66        |79.56      |40.24        |
+--------------------------------+-------------+-----+-------------+-----------+-------------+
only showing top 5 rows



# Check order payments table regarding dropna and dropDuplicates and write it in silver layer.

In [21]:
# for order payments, decided to aggregate the payment installments and value per order, after checking the id and installments for null value and 
# duplications. there will be 2 table, one order_payments after the transformations and order_payments_agg from the previous one
# One row per PAYMENT - the cleansed source entity
silver_payments = (
    read_bronze("order_payments")
    .withColumn("payment_installments", F.col("payment_installments").cast(T.IntegerType()))
    .withColumn("payment_value", F.col("payment_value").cast(T.DoubleType()))
    .dropna(subset=["order_id", "payment_sequential"])
    .dropDuplicates(["order_id", "payment_sequential"])
)

silver_payments = write_silver(silver_payments, "order_payments", partitioned=False)

order_payments         rows= 103,886 partitioned=False


In [22]:
# this table has one row per order, will be joined with the enriched order table
silver_payments_agg = (
    silver_payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_installments").alias("payment_count"),
        F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
        F.count("*").alias("payment_record_count"),
    )
)

silver_payments_agg = write_silver(silver_payments_agg, "order_payments_agg", partitioned=False)
silver_payments_agg.show(5) # some examples

order_payments_agg     rows=  99,440 partitioned=False
+--------------------+-------------+-------------------+--------------------+
|            order_id|payment_count|total_payment_value|payment_record_count|
+--------------------+-------------+-------------------+--------------------+
|298fcdf1f73eb413e...|            2|              96.12|                   1|
|c39414c195d0f94c9...|            2|             139.22|                   1|
|fa2ea4b6e84c1c0fc...|            1|             227.12|                   1|
|85a4fbdba48c81592...|            3|              34.09|                   1|
|d121cbcb7091fbd55...|            3|               49.7|                   1|
+--------------------+-------------+-------------------+--------------------+
only showing top 5 rows



# Check order reviews table regarding dropna and dropDuplicates and write it in silver layer.

In [23]:
silver_reviews = (
    read_bronze("order_reviews")
    .withColumn("review_score", F.col("review_score").cast(T.IntegerType()))
    .withColumn("review_creation_date", F.col("review_creation_date").cast(T.TimestampType()))
    .dropna(
        subset=["review_id", "order_id", "review_creation_date"])
    .dropDuplicates(["review_id", "order_id"])
    .withColumn("year", F.year("review_creation_date"))
    .withColumn("month", F.month("review_creation_date"))
    .withColumn("day", F.dayofmonth("review_creation_date"))
)

silver_reviews = write_silver(
    silver_reviews,
    "order_reviews",
)

order_reviews          rows=  99,224 partitioned=True


# Join different table to create an enriched table named orders_enriched.

In [24]:
# remove the year, month, day columns to not get duplicated
items_for_join = silver_order_items.drop(*PARTITION_COLS)

silver_enriched = (
    silver_orders
    .join(items_for_join, on="order_id", how="inner") # join with items; from one row per order to one row per order line
     # left joins to include columns from each table
    .join(silver_wr["products"], on="product_id", how="left")
    .join(silver_wr["sellers"], on="seller_id", how="left")
    .join(silver_wr["customers"], on="customer_id", how="left")
    .join(silver_payments_agg, on="order_id", how="left")
    # handle unmatched values
    .withColumn("payment_count", F.coalesce(F.col("payment_count"), F.lit(0)))
    .withColumn("payment_record_count", F.coalesce(F.col("payment_record_count"), F.lit(0)))
    .withColumn("total_payment_value", F.coalesce(F.col("total_payment_value"), F.lit(0.0)))
)
# here are included also the 4 requested calculated columns: total_price, delivery_time_days, payment_count, profit_margin
silver_enriched = silver_enriched.select(
    "order_id",
    "order_item_id",
    "customer_id",
    "product_id",
    "seller_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "delivery_time_days",
    "price",
    "freight_value",
    "total_price",
    "profit_margin",
    "payment_count",
    "payment_record_count",
    "total_payment_value",
    "product_category_name",
    "customer_city",
    "customer_state",
    "seller_city",
    "seller_state",
    "year",
    "month",
    "day",
)

silver_enriched = write_silver(
    silver_enriched,
    "order_enriched",
)

silver_enriched.select(
    "order_id",
    "product_category_name",
    "customer_state",
    "total_price",
    "profit_margin",
    "delivery_time_days",
    "payment_count",
).show(10, truncate=False)

order_enriched         rows= 112,650 partitioned=True
+--------------------------------+---------------------------+--------------+-----------+-------------+------------------+-------------+
|order_id                        |product_category_name      |customer_state|total_price|profit_margin|delivery_time_days|payment_count|
+--------------------------------+---------------------------+--------------+-----------+-------------+------------------+-------------+
|000229ec398224ef6ca0657da4fc703e|moveis_decoracao           |MG            |216.87     |181.13       |8                 |5            |
|00042b26cf59d7ce69dfabb4e55b4fd9|ferramentas_jardim         |SP            |218.04     |181.76       |25                |3            |
|0005f50442cb953dcd1d21e1fb923495|livros_tecnicos            |SP            |65.39      |42.59        |2                 |1            |
|00061f2a7bc09da83e415a52dc8a4af1|beleza_saude               |SP            |68.87      |51.11        |5                 |3 

# Do a quality check for the enriched table for every category.

In [25]:
enriched_from_delta = (
    spark.read
    .format("delta")
    .load(str(SILVER_DIR / "order_enriched"))
)

quality_summary = enriched_from_delta.select(
    F.count("*").alias("count_enriched_rows"),
    F.countDistinct("order_id").alias("count_distinct_orders"),
    F.sum(
        F.when(F.col("product_category_name").isNull(), 1).otherwise(0)
    ).alias("null_categories"),
    F.sum(
        F.when(F.col("total_price").isNull(), 1).otherwise(0)
    ).alias("null_total_price"),
    F.sum(
        F.when(F.col("total_price") < 0, 1).otherwise(0)
    ).alias("negative_total_price"),
    F.sum(
        F.when(F.col("delivery_time_days") < 0, 1).otherwise(0)
    ).alias("negative_delivery_days"),
)

quality_summary.show(truncate=False)


+-------------------+---------------------+---------------+----------------+--------------------+----------------------+
|count_enriched_rows|count_distinct_orders|null_categories|null_total_price|negative_total_price|negative_delivery_days|
+-------------------+---------------------+---------------+----------------+--------------------+----------------------+
|112650             |98666                |0              |0               |0                   |0                     |
+-------------------+---------------------+---------------+----------------+--------------------+----------------------+



In [26]:
spark.stop()